# 02 — Build Self-Generated Reasoning Preferences (Condition B)

**Notebook 2 of 7.** Constructs the preference dataset that Notebook 04 will train DPO on.

For each GSM8K **train** problem (never test — we don't touch the eval set), sample K=4 chains-of-thought from the base model, then pair a *correct* chain (matches ground truth) with an *incorrect* chain to form one DPO training example. Target: 500 pairs.

- **Runtime:** ~3 hours on Colab free-tier T4. Resumable across sessions.
- **Outputs on Drive:**
  - `data/self_generated_prefs.jsonl` — one raw pair per line (audit trail + DPO input)
  - `data/self_generated_prefs.processed.log` — indices of problems already attempted (resume marker)
  - `data/self_generated_prefs.meta.json` — sampling config and yield statistics

Before you start: **Runtime → Change runtime type → T4 GPU**.

## 1. Install dependencies

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Mount Drive and set paths

Use the **same** `PROJECT_ROOT` as Notebook 01 so everything ends up in one place.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/dpo-reasoning-transfer'
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)

PREF_PATH = f'{PROJECT_ROOT}/data/self_generated_prefs.jsonl'
PROC_LOG_PATH = f'{PROJECT_ROOT}/data/self_generated_prefs.processed.log'
META_PATH = f'{PROJECT_ROOT}/data/self_generated_prefs.meta.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'Output JSONL:  {PREF_PATH}')

## 3. Load `src/`

In [ ]:
import sys
CODE_DIR = PROJECT_ROOT   # or f'{PROJECT_ROOT}/repo' if you've cloned from GitHub
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.prefs import (
    build_reasoning_preference_set,
    load_preference_jsonl,
    format_pair_for_dpo,
)
print('src/prefs.py loaded')

## 4. Load the base model

**Must be identical to Notebook 01** — same weights, same quantization. Otherwise the sampled chains don't represent "the model we're about to DPO on".

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
print(f'Model device: {next(model.parameters()).device}')

## 5. Design of the preference set

Choices made in `src/prefs.py`, spelled out here so a reader of the report doesn't have to dig through code:

**Sampling.** K=4 chains per problem, temperature 0.8, top-p 0.95, cap 384 new tokens. At the base model's ~60% GSM8K accuracy, this gives a ~85% chance that any given problem produces at least one correct AND one incorrect chain (verified analytically and by simulation: `1 - 0.6⁴ - 0.4⁴ ≈ 0.848`).

**Pair selection.** From the K chains: `chosen` = uniformly random correct, `rejected` = uniformly random incorrect. Deliberately **not** "shortest correct, longest incorrect" — that would inject an anti-length signal and confound the reasoning-signal hypothesis. Any length imbalance is instead logged and discussed as a limitation.

**Correctness definition.** Extracted-final-answer match against ground truth (the same `answers_match` used in the evaluator). A chain labeled "correct" may still contain broken reasoning that happens to arrive at the right number ("spurious correctness"). This is a known limitation of final-answer verifiers; process-reward alternatives are noted as future work.

**Source and contamination.** Sampling is exclusively over GSM8K **train**. The 500-problem test slice used by Notebook 01 is never touched here.

In [ ]:
# --- Sampling configuration ---
K              = 4       # samples per problem
TEMPERATURE    = 0.8
TOP_P          = 0.95
MAX_NEW_TOKENS = 384

# --- Dataset scoping ---
TARGET_N_PAIRS       = 500      # what Notebook 04 will train on
MAX_SOURCE_PROBLEMS  = 700      # upper bound on train problems to sample from
SAMPLING_SEED        = 42

print(f'K={K}, temp={TEMPERATURE}, top_p={TOP_P}')
print(f'Target {TARGET_N_PAIRS} pairs from up to {MAX_SOURCE_PROBLEMS} train problems')

## 6. Load GSM8K train

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset(
    'openai/gsm8k', 'main',
    split=f'train[:{MAX_SOURCE_PROBLEMS}]',
)
print(f'Loaded {len(train_dataset)} train problems')
print(f'Example question:\n  {train_dataset[0]["question"][:200]}...')

## 7. Run the sampling loop

Resumable: appends each attempted problem to `processed.log` and each yielded pair to `self_generated_prefs.jsonl`. Re-run this cell after a Colab drop; it picks up where it left off. Stops early once `TARGET_N_PAIRS` is reached.

Progress bar shows running pair count and yield rate.

In [ ]:
metadata = build_reasoning_preference_set(
    model=model,
    tokenizer=tokenizer,
    source_dataset=train_dataset,
    output_jsonl_path=PREF_PATH,
    processed_log_path=PROC_LOG_PATH,
    metadata_path=META_PATH,
    target_n_pairs=TARGET_N_PAIRS,
    max_source_problems=MAX_SOURCE_PROBLEMS,
    K=K,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_new_tokens=MAX_NEW_TOKENS,
    seed=SAMPLING_SEED,
)

print()
print('=' * 60)
print(f"Attempted: {metadata['yield']['n_processed']} problems")
print(f"Yielded:   {metadata['yield']['n_pairs']} pairs "
      f"({metadata['yield']['yield_rate']:.1%})")
print('=' * 60)

## 8. Yield statistics and length audit

Two things we care about for the report:
1. Did we actually get the target number of pairs?
2. Are `chosen` and `rejected` roughly length-matched? (A large gap would mean DPO learns "prefer shorter/longer" rather than "prefer correct reasoning".)

In [ ]:
import statistics

pairs = load_preference_jsonl(PREF_PATH)
print(f'Loaded {len(pairs)} preference pairs')

# Length audit
chosen_lens   = [p['chosen_length_chars'] for p in pairs]
rejected_lens = [p['rejected_length_chars'] for p in pairs]
delta_lens    = [c - r for c, r in zip(chosen_lens, rejected_lens)]

print(f"\nResponse length (chars):")
print(f"  chosen:    mean={statistics.mean(chosen_lens):.0f}, "
      f"median={statistics.median(chosen_lens):.0f}")
print(f"  rejected:  mean={statistics.mean(rejected_lens):.0f}, "
      f"median={statistics.median(rejected_lens):.0f}")
print(f"  chosen - rejected: mean={statistics.mean(delta_lens):+.0f}, "
      f"median={statistics.median(delta_lens):+.0f}")

# How balanced was the sampling? (Per-problem correct/incorrect ratio)
correct_counts   = [p['n_correct_samples']   for p in pairs]
incorrect_counts = [p['n_incorrect_samples'] for p in pairs]
print(f"\nPer-problem sample balance (of K={pairs[0]['n_correct_samples'] + pairs[0]['n_incorrect_samples']}):")
print(f"  mean correct per problem:   {statistics.mean(correct_counts):.2f}")
print(f"  mean incorrect per problem: {statistics.mean(incorrect_counts):.2f}")

## 9. Sanity check — inspect two example pairs

Read a couple manually. Look for: (a) `chosen` really is arithmetically right, (b) `rejected` really is wrong, (c) both look like plausible attempts.

In [ ]:
for i, p in enumerate(pairs[:2]):
    print(f"\n{'=' * 70}")
    print(f"PAIR {i}   (source train problem #{p['problem_index']})")
    print('=' * 70)
    print(f"Q: {p['question'][:220]}..." if len(p['question']) > 220 else f"Q: {p['question']}")
    print(f"GT: {p['gt_answer']}")
    print(f"K sampled answers: {p['all_sample_answers']}")
    print(f"\n--- CHOSEN (pred = {p['chosen_answer']}) ---")
    print(p['chosen'][:500] + ('...' if len(p['chosen']) > 500 else ''))
    print(f"\n--- REJECTED (pred = {p['rejected_answer']}) ---")
    print(p['rejected'][:500] + ('...' if len(p['rejected']) > 500 else ''))

## 10. Verify DPO-format conversion

Notebook 04 will load the raw JSONL and apply `format_pair_for_dpo` at training time. Do a dry-run here to catch any tokenizer / template issues *now*, not during a 30-minute training job.

In [ ]:
example_formatted = format_pair_for_dpo(pairs[0], tokenizer)
print('Keys:', list(example_formatted.keys()))
print()
print('--- prompt (first 400 chars) ---')
print(example_formatted['prompt'][:400])
print()
print('--- chosen (first 200 chars) ---')
print(example_formatted['chosen'][:200])
print()
print('--- rejected (first 200 chars) ---')
print(example_formatted['rejected'][:200])

# Length sanity — DPOConfig max_length=1024, max_prompt_length=512
p_tok = tokenizer(example_formatted['prompt'], return_tensors='pt')['input_ids'].shape[1]
c_tok = tokenizer(example_formatted['chosen'], return_tensors='pt')['input_ids'].shape[1]
r_tok = tokenizer(example_formatted['rejected'], return_tensors='pt')['input_ids'].shape[1]
print(f"\nToken counts — prompt: {p_tok} (limit 512), chosen: {c_tok}, rejected: {r_tok}, "
      f"total_chosen: {p_tok+c_tok}, total_rejected: {p_tok+r_tok} (limit 1024)")

## 11. Distribution of pair lengths across the dataset

Quick histogram of chosen-vs-rejected length deltas. A tight distribution around zero is what we want; a strong skew would motivate length-controlled analysis in the report.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist([chosen_lens, rejected_lens],
             bins=25, label=['chosen', 'rejected'],
             color=['#4C72B0', '#DD8452'], alpha=0.7)
axes[0].set_xlabel('Response length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Chosen vs. rejected length distributions')
axes[0].legend()

axes[1].hist(delta_lens, bins=30, color='#55A868', alpha=0.8)
axes[1].axvline(0, color='k', ls='--', lw=1)
axes[1].axvline(statistics.mean(delta_lens), color='r', ls='-', lw=1,
                label=f'mean = {statistics.mean(delta_lens):+.0f}')
axes[1].set_xlabel('chosen_length - rejected_length (chars)')
axes[1].set_ylabel('Count')
axes[1].set_title('Length delta (positive = chosen longer)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/results/figures/preference_length_audit.png',
            dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved figure to results/figures/preference_length_audit.png')

## Next

- `data/self_generated_prefs.jsonl` on Drive is the Condition B training set. Notebook 04 will consume it directly.
- Metadata (sampling config, yield rate, seed) is in `data/self_generated_prefs.meta.json` — cite these numbers in the report.
- **Next up:** `03_dpo_A_ultrafeedback.ipynb` — train DPO on the general-purpose baseline preference set (Condition A), then eval on GSM8K[:500]. Then `04_dpo_B_reasoning.ipynb` does the same for the set we just built.